# MACE foundation models: structure relaxation and molecular dynamics

This notebook is a clean and generic version of the original `MACEfoundation_test` notebook. It lets you load a local MACE checkpoint, choose the appropriate head, relax a structure, run MD, and save the dynamics in `.xyz` format.

Useful references:

- https://github.com/ACEsuit/mace-foundations
- https://huggingface.co/mace-foundations/mace-mh-1

![MACE-MH-1 overview](https://cdn-uploads.huggingface.co/production/uploads/630df4308df86f1e5bec63c3/sqXxh5DrLSXj8elKnJJS8.png)


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models").exists() and (PROJECT_ROOT.parent / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib"))
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

import matplotlib.pyplot as plt
import numpy as np
from ase import units
from ase.build import molecule
from ase.io import Trajectory, read, write
from ase.md.nvtberendsen import NVTBerendsen
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase.md.verlet import VelocityVerlet
from ase.optimize import BFGS
from ase.visualize import view
from mace.calculators import mace_mp
from tqdm.auto import tqdm
from IPython.display import display
try:
    import nglview as nv
    HAVE_NGLVIEW = True
except Exception:
    HAVE_NGLVIEW = False

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

try:
    from ase.md.nose_hoover_chain import NoseHooverChainNVT
    HAVE_NHC = True
except Exception:
    HAVE_NHC = False

FOUNDATION_HEADS = {
    "omat_pbe": "General materials.",
    "oc20_usemppbe": "Surfaces and adsorbates.",
    "rgd1_b3lyp": "Reaction-focused chemistry.",
    "omol": "Organic molecules.",
    "spice_wB97M": "Molecular benchmarking.",
}

def list_local_models(models_dir=MODELS_DIR):
    return sorted(Path(models_dir).glob("*.model"))

def create_mace_calculator(model_path, head=None, device="cpu", default_dtype="float64", dispersion=True):
    kwargs = {
        "model": str(model_path),
        "device": device,
        "default_dtype": default_dtype,
        "dispersion": dispersion,
    }
    if head is not None:
        kwargs["head"] = head
    try:
        return mace_mp(**kwargs)
    except RuntimeError as exc:
        if dispersion and "torch-dftd" in str(exc):
            print("torch-dftd not found; retrying with dispersion=False")
            kwargs["dispersion"] = False
            return mace_mp(**kwargs)
        raise

def optimize_structure(atoms, fmax=0.05, max_steps=200, trajectory_path=None, desc="Optimization", show_progress=True):
    if trajectory_path is not None:
        trajectory_path = Path(trajectory_path)
        trajectory_path.parent.mkdir(parents=True, exist_ok=True)
        optimizer = BFGS(atoms, logfile=None, trajectory=str(trajectory_path))
    else:
        optimizer = BFGS(atoms, logfile=None)
    progress = tqdm(desc=desc, unit="step", disable=not show_progress)
    converged = False
    try:
        for converged in optimizer.irun(fmax=fmax, steps=max_steps):
            progress.update(1)
            if converged:
                break
    finally:
        progress.close()
    if not converged:
        print(f"Warning: {desc} reached the step limit ({max_steps}) before convergence.")
    return atoms

def has_meaningful_velocities(atoms):
    velocities = atoms.get_velocities()
    return velocities is not None and np.all(np.isfinite(velocities)) and np.linalg.norm(velocities) > 1e-8


def is_periodic_system(atoms):
    return bool(np.any(atoms.pbc))

def run_md(atoms, timestep_fs, nsteps, temperature_K=None, traj_path=OUTPUTS_DIR / "md.traj", log_path=OUTPUTS_DIR / "md.log", traj_interval=1, log_interval=10, progress_desc="MD"):
    traj_path = Path(traj_path)
    log_path = Path(log_path)
    traj_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    if has_meaningful_velocities(atoms):
        dyn = VelocityVerlet(atoms, timestep_fs * units.fs)
        md_mode = "NVE"
    else:
        if temperature_K is None:
            raise ValueError("temperature_K is required when the structure has no velocities.")
        MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
        Stationary(atoms)
        if not is_periodic_system(atoms):
            ZeroRotation(atoms)
        if HAVE_NHC:
            dyn = NoseHooverChainNVT(atoms, timestep=timestep_fs * units.fs, temperature_K=temperature_K, tchain=3, tdamp=100.0)
            md_mode = "NVT (Nose-Hoover chain)"
        else:
            dyn = NVTBerendsen(atoms, timestep_fs * units.fs, temperature_K=temperature_K, taut=100.0 * units.fs)
            md_mode = "NVT (Berendsen fallback)"

    traj = Trajectory(str(traj_path), "w", atoms)
    log_path.write_text("")

    def log_step():
        epot = atoms.get_potential_energy()
        ekin = atoms.get_kinetic_energy()
        temp = ekin / (1.5 * units.kB * len(atoms))
        with log_path.open("a", encoding="utf-8") as handle:
            handle.write(f"{dyn.nsteps:6d}  Epot {epot: .8f}  Ekin {ekin: .8f}  Etot {epot + ekin: .8f}  T {temp: .2f}\\n")

    dyn.attach(traj.write, interval=traj_interval)
    dyn.attach(log_step, interval=log_interval)

    progress = tqdm(total=nsteps, desc=f"{progress_desc} [{md_mode}]", unit="step")
    try:
        for _ in range(nsteps):
            dyn.run(1)
            progress.update(1)
    finally:
        progress.close()
        traj.close()
    return md_mode, traj_path, log_path

def export_xyz(traj_path, xyz_path):
    traj_path = Path(traj_path)
    xyz_path = Path(xyz_path)
    xyz_path.parent.mkdir(parents=True, exist_ok=True)
    frames = Trajectory(str(traj_path))
    write(str(xyz_path), frames, format="xyz")



def show_atoms(atoms, label="structure"):
    try:
        obj = view(atoms, viewer="x3d")
        if obj is not None:
            display(obj)
        return obj
    except Exception as exc:
        fallback = OUTPUTS_DIR / f"{label}.xyz"
        write(fallback, atoms)
        print(f"Visualization fallback written to {fallback} ({exc})")
        return None


def make_interactive_trajectory_viewer(traj_path, label="trajectory"):
    traj_path = Path(traj_path)
    if not traj_path.exists():
        print(f"Trajectory file not found: {traj_path}")
        return None
    if not HAVE_NGLVIEW:
        print("nglview is not available; install it to get an interactive trajectory player.")
        print(f"Trajectory saved at: {traj_path}")
        return None
    traj = Trajectory(str(traj_path))
    if len(traj) == 0:
        print(f"No frames in trajectory: {traj_path}")
        return None
    view_widget = nv.show_asetraj(traj)
    try:
        view_widget.clear_representations()
        view_widget.add_ball_and_stick()
        view_widget.center()
        view_widget.layout.width = "100%"
        view_widget.layout.height = "600px"
        if hasattr(view_widget, "player"):
            try:
                view_widget.player.step = 1
                view_widget.player.delay = 100
                view_widget.player.sync_frame = True
            except Exception:
                pass
    except Exception:
        pass
    return view_widget

def parse_md_log(log_path):
    steps, epot, ekin, etot, temp = [], [], [], [], []
    text = Path(log_path).read_text()
    text = text.replace("\\n", "\n")
    for line in text.splitlines():
        parts = line.split()
        if not parts:
            continue
        steps.append(int(parts[0]))
        epot.append(float(parts[2]))
        ekin.append(float(parts[4]))
        etot.append(float(parts[6]))
        temp.append(float(parts[8]))
    return {"steps": np.asarray(steps), "epot": np.asarray(epot), "ekin": np.asarray(ekin), "etot": np.asarray(etot), "temp": np.asarray(temp)}


## Local models

Place one or more `.model` files in `../models/`.


In [ ]:
for model_path in list_local_models():
    print(model_path)


## Load a MACE model


In [ ]:
MODEL_PATH = MODELS_DIR / "mace-mh-1.model"   # Local MACE checkpoint to load.
MODEL_HEAD = "omat_pbe"                        # Head to use for a multi-head foundation model.
DEVICE = "cpu"                                 # 'cpu' or 'cuda'.
DEFAULT_DTYPE = "float64"                      # float64 is slower but safer for optimization.
USE_DISPERSION = True                          # Use D3 dispersion through torch-dftd if available.

for key, description in FOUNDATION_HEADS.items():
    print(f"{key:16s} : {description}")

calc = create_mace_calculator(MODEL_PATH, head=MODEL_HEAD, device=DEVICE, default_dtype=DEFAULT_DTYPE, dispersion=USE_DISPERSION)
calc


## Optional water-dimer sanity check


In [ ]:
RUN_WATER_DIMER_TEST = False                               # Run a small sanity check before using a larger system.
WATER_DIMER_VISUALIZE = True                              # Show the optimized water dimer in the notebook.
WATER_DIMER_SAVE_OPTIMIZED = False                        # Save the optimized water dimer to disk.
WATER_DIMER_OPTIMIZED_PATH = OUTPUTS_DIR / "water_dimer_optimized.xyz"  # Output file for the test geometry.

if RUN_WATER_DIMER_TEST:
    atoms_test = molecule("H2O") + molecule("H2O")
    atoms_test[3:].translate([3.0, 0.0, 0.0])
    atoms_test.calc = calc
    print(f"Initial energy: {atoms_test.get_potential_energy():.6f} eV")
    optimize_structure(atoms_test, fmax=0.02, desc="Water dimer optimization")
    if WATER_DIMER_SAVE_OPTIMIZED:
        write(WATER_DIMER_OPTIMIZED_PATH, atoms_test)
        print(f"Wrote {WATER_DIMER_OPTIMIZED_PATH}")
    print(f"Optimized energy: {atoms_test.get_potential_energy():.6f} eV")
    if WATER_DIMER_VISUALIZE:
        show_atoms(atoms_test, label="water_dimer")


## Application to a structure of interest

You can build a small molecular system directly or load a structure file.


In [ ]:
STRUCTURE_SOURCE = "build"                                # 'build' to generate a small test system, 'load' to read a file.

# Option 1: build directly with ASE
ASE_MOLECULE_NAME = "H2O"                                # ASE molecule name used when STRUCTURE_SOURCE='build'.
MAKE_DIMER = True                                         # Build a simple dimer by duplicating the molecule.

# Option 2: load from file
STRUCTURE_PATH = Path("path/to/your/structure.xyz")      # Input file path when STRUCTURE_SOURCE='load'.
STRUCTURE_FORMAT = None                                   # Force the file format, or None for ASE auto-detection.

RUN_GEOMETRY_OPTIMIZATION = True                          # Relax the structure before MD.
OPT_FMAX = 0.05                                           # Force convergence threshold for the optimization (eV/A).
OPT_MAX_STEPS = 200                                       # Maximum number of optimization steps.
WRITE_OPTIMIZED_STRUCTURE = True                          # Save the optimized structure if optimization is run.
OPTIMIZED_STRUCTURE_PATH = OUTPUTS_DIR / "optimized_structure.xyz"  # Output file for the optimized structure.
SHOW_OPTIMIZED_STRUCTURE = True                           # Visualize the optimized geometry in the notebook.


In [ ]:
if STRUCTURE_SOURCE == "build":
    atoms = molecule(ASE_MOLECULE_NAME)
    if MAKE_DIMER:
        other = molecule(ASE_MOLECULE_NAME)
        other.translate([3.0, 0.0, 0.0])
        atoms = atoms + other
elif STRUCTURE_SOURCE == "load":
    atoms = read(STRUCTURE_PATH, format=STRUCTURE_FORMAT) if STRUCTURE_FORMAT else read(STRUCTURE_PATH)
else:
    raise ValueError("STRUCTURE_SOURCE must be 'build' or 'load'.")

atoms.calc = calc
energy0 = atoms.get_potential_energy()
forces0 = atoms.get_forces()
velocities0 = atoms.get_velocities()
has_velocities0 = velocities0 is not None and np.linalg.norm(velocities0) > 1e-8

print(atoms)
print(f"Initial potential energy: {energy0:.8f} eV")
print(f"Forces array shape       : {forces0.shape}")
print(f"Initial velocities found : {has_velocities0}")

if RUN_GEOMETRY_OPTIMIZATION:
    optimize_structure(atoms, fmax=OPT_FMAX, max_steps=OPT_MAX_STEPS, desc="Structure optimization")
    if WRITE_OPTIMIZED_STRUCTURE:
        OPTIMIZED_STRUCTURE_PATH.parent.mkdir(parents=True, exist_ok=True)
        write(OPTIMIZED_STRUCTURE_PATH, atoms)
        print(f"Wrote {OPTIMIZED_STRUCTURE_PATH}")
    if SHOW_OPTIMIZED_STRUCTURE:
        show_atoms(atoms, label="optimized_structure")


## Molecular dynamics

The trajectory is saved in ASE `.traj` format and exported to `.xyz`.


In [ ]:
TIMESTEP_FS = 1.0                                  # MD time step in femtoseconds.
NSTEPS = 500                                      # Maximum number of MD steps.
TEMPERATURE_K = 300.0                             # Temperature used if velocities must be initialized.
TRAJ_INTERVAL = 1                                 # Write one trajectory frame every TRAJ_INTERVAL steps.
LOG_INTERVAL = 10                                 # Write one log line every LOG_INTERVAL steps.
XYZ_TRAJECTORY_PATH = OUTPUTS_DIR / "mace_md.xyz"  # Exported XYZ trajectory path.
SHOW_FINAL_STRUCTURE = True                       # Visualize the last MD frame.

md_mode, traj_path, log_path, steps_run, stopped_early, stop_reason = run_md(
    atoms,
    timestep_fs=TIMESTEP_FS,
    nsteps=NSTEPS,
    temperature_K=TEMPERATURE_K,
    traj_path=OUTPUTS_DIR / "mace_md.traj",
    log_path=OUTPUTS_DIR / "mace_md.log",
    traj_interval=TRAJ_INTERVAL,
    log_interval=LOG_INTERVAL,
    progress_desc="Production MD",
)
export_xyz(traj_path, XYZ_TRAJECTORY_PATH)
print(f"MD mode       : {md_mode}")
print(f"Trajectory    : {traj_path}")
print(f"XYZ trajectory: {XYZ_TRAJECTORY_PATH}")
print(f"Log file      : {log_path}")
print(f"Steps run     : {steps_run}")


## Final structure and plots


In [ ]:
traj = Trajectory(str(traj_path))
last_frame = traj[-1]
log = parse_md_log(log_path)

fig, axes = plt.subplots(2, 2, figsize=(9, 7), dpi=140, sharex=True)
(ax1, ax2), (ax3, ax4) = axes
ax1.plot(log["steps"], log["epot"])
ax1.set_ylabel("Potential energy (eV)")
ax1.set_title("Epot")
ax2.plot(log["steps"], log["ekin"])
ax2.set_ylabel("Kinetic energy (eV)")
ax2.set_title("Ekin")
ax3.plot(log["steps"], log["etot"])
ax3.set_xlabel("Step")
ax3.set_ylabel("Total energy (eV)")
ax3.set_title("Etot")
ax4.plot(log["steps"], log["temp"])
ax4.set_xlabel("Step")
ax4.set_ylabel("Temperature (K)")
ax4.set_title("Temperature")
fig.suptitle("MD log summary", y=0.98)
fig.tight_layout()
print(f"Logged MD points: {len(log['steps'])}")


## Interactive trajectory

If `nglview` is installed, this cell opens an interactive trajectory player directly in the notebook, with play/pause controls and a frame slider.


In [ ]:
make_interactive_trajectory_viewer(traj_path, label="md")
